In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/cleaned-foodprox/fpx_clean.csv
/kaggle/input/cleaned-off/off_filtered.csv


In [2]:
fpx_path='/kaggle/input/datasets/raesiow/cleaned-foodprox/fpx_clean.csv'
off_path='/kaggle/input/cleaned-off/off_filtered.csv'
fpx = pd.read_csv(fpx_path, dtype={"code": str})
off = pd.read_csv(off_path, dtype={"code": str})  
print(fpx.shape, off.shape)


(8238, 20) (71298, 18)


In [3]:
!pip install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.1 MB/s eta 0:00:0000:0100:01


In [4]:
import numpy as np, time
from rapidfuzz import process, fuzz

THRESH = 70
pairs = []          # will hold (fpx_name, off_name, score)
off_names = off["name_clean"].unique()
fpx_names = fpx["name_clean"].unique()

t = time.time()
for start in range(0, len(fpx_names), 200):
    chunk = fpx_names[start:start + 200]
    a = process.cdist(chunk, off_names, scorer=fuzz.token_set_ratio,  dtype=np.uint8, workers=-1)
    b = process.cdist(chunk, off_names, scorer=fuzz.token_sort_ratio, dtype=np.uint8, workers=-1)
    score = (a.astype(np.float32) + b) / 2

    rows, cols = np.where(score >= THRESH)       # positions that pass the cutoff
    for r, c in zip(rows, cols):
        pairs.append((chunk[r], off_names[c], score[r, c]))

    if (start // 200) % 10 == 0:
        print(start, "done,", len(pairs), "pairs so far,", round(time.time() - t), "s")

cand = pd.DataFrame(pairs, columns=["fpx_name", "off_name", "score"])
print(cand.shape)
print(cand["fpx_name"].nunique(), "of", len(fpx_names), "fpx names have at least one candidate")
cand.to_csv("/kaggle/working/cand_70.csv", index=False)

0 done, 2173 pairs so far, 8 s
2000 done, 17226 pairs so far, 84 s
4000 done, 21472 pairs so far, 174 s
6000 done, 47197 pairs so far, 252 s
8000 done, 70260 pairs so far, 317 s
(70260, 3)
4852 of 8112 fpx names have at least one candidate


In [ ]:
!pip install sentence-transformers

In [6]:
import torch, time
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
off_names = off["name_clean"].unique()
fpx_names = fpx["name_clean"].unique()
t = time.time()
fpx_emb = model.encode(list(fpx_names), batch_size=256, normalize_embeddings=True,
                       convert_to_tensor=True, show_progress_bar=False)
off_emb = model.encode(list(off_names), batch_size=256, normalize_embeddings=True,
                       convert_to_tensor=True, show_progress_bar=False)
print(fpx_emb.shape, off_emb.shape, round(time.time() - t), "s")

device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

torch.Size([8112, 384]) torch.Size([53884, 384]) 8 s


In [7]:
sims = fpx_emb @ off_emb.T            # 8112 x 53884
v, ix = sims.topk(20, dim=1)
v, ix = v.cpu().numpy(), ix.cpu().numpy()
del sims                               # free the 1.75 GB straight away

FLOOR = 0.60
r_idx, k_idx = np.where(v >= FLOOR)    # positions that pass the floor
cand_m = pd.DataFrame({
    "fpx_name": fpx_names[r_idx],
    "off_name": off_names[ix[r_idx, k_idx]],
    "cos": v[r_idx, k_idx],
})
print(cand_m.shape, cand_m["fpx_name"].nunique(), "of", len(fpx_names), "foods have candidates")
cand_m.to_csv('/kaggle/working/can_minilm.csv', index = False)

(108763, 3) 7267 of 8112 foods have candidates


In [8]:
cand_u = cand[["fpx_name", "off_name", "score"]].merge(
    cand_m[["fpx_name", "off_name", "cos"]],
    on=["fpx_name", "off_name"], how="outer")
cand_u["in_rf"] = cand_u["score"].notna()
cand_u["in_mini"] = cand_u["cos"].notna()
print(cand_u.shape)
print(cand_u[["in_rf", "in_mini"]].value_counts())

(156146, 6)
in_rf  in_mini
False  True       85886
True   False      47383
       True       22877
Name: count, dtype: int64


In [9]:
nut = ["protein_g", "fat_g", "carb_excl_fiber_g", "sugars_g", "fiber_g", "sat_fat_g", "sodium_g"]
f = fpx[["code", "name_clean", "nova"] + nut].rename(
    columns={"code": "fpx_code", "name_clean": "fpx_name", **{c: c + "_f" for c in nut}})
o = off[["code", "name_clean"] + nut].rename(
    columns={"code": "off_code", "name_clean": "off_name", **{c: c + "_o" for c in nut}})

pp2 = cand_u.merge(f, on="fpx_name").merge(o, on="off_name")
print(pp2.shape)

(296922, 23)


In [10]:
tol = {"protein_g": 1.5, "fat_g": 1.5, "carb_excl_fiber_g": 2.0, "sugars_g": 2.0,
       "fiber_g": 1.0, "sat_fat_g": 1.0, "sodium_g": 0.1}
required = ["protein_g", "fat_g", "carb_excl_fiber_g", "sugars_g",
            "fiber_g", "sat_fat_g", "sodium_g"]

def close(df, n):
    a, b = df[n + "_f"], df[n + "_o"]
    return (a - b).abs() <= np.maximum(tol[n], 0.25 * np.maximum(a, b))

pp2["nut_ok"] = True
for n in required:
    pp2["nut_ok"] &= close(pp2, n)

print("passing pairs:", pp2["nut_ok"].sum())            # expect 22344
print("foods with a passing pair:", pp2.loc[pp2["nut_ok"], "fpx_code"].nunique())   # expect 3076

passing pairs: 22344
foods with a passing pair: 3076


In [11]:
def run_method(pp_in):
    ok = pp_in[pp_in["nut_ok"]].copy()
    d = 0
    for n in required:
        a, b = ok[n + "_f"], ok[n + "_o"]
        d = d + (a - b).abs() / np.maximum(np.maximum(a, b), 1.0)
    ok["nut_dist"] = d
    ok["rank"] = ok["score"].fillna(0) / 100 + ok["cos"].fillna(0)
    ok = ok.sort_values(["fpx_code", "rank", "nut_dist"], ascending=[True, False, True])
    best = ok.drop_duplicates("fpx_code")
    best = best.sort_values(["rank", "nut_dist"], ascending=[False, True])
    return best.drop_duplicates("off_code")


In [12]:
for label, sub in [("rapidfuzz only", pp2[pp2["in_rf"]]),
                   ("MiniLM only",     pp2[pp2["in_mini"]]),
                   ("union",           pp2)]:
    res = run_method(sub)
    print(label, len(res), res["nova"].value_counts().sort_index().tolist())

rapidfuzz only 1193 [288, 51, 170, 684]
MiniLM only 1781 [462, 61, 242, 1016]
union 1884 [477, 61, 265, 1081]


In [13]:
res = run_method(pp2)
res["source"] = np.where(res["in_rf"] & res["in_mini"], "both", 
                 np.where(res["in_rf"], "rf_only", "mini_only"))
print("off_code unique:", res["off_code"].is_unique)

keep = ["fpx_code", "fpx_name", "nova", "off_code", "off_name", "source", "score", "cos",
        "protein_g_f", "fat_g_f", "carb_excl_fiber_g_f", "sugars_g_f",
        "fiber_g_f", "sat_fat_g_f", "sodium_g_f"]
matched = res[keep].merge(
    off[["code", "ingredients_text", "categories_en", "pnns_groups_2", "image_url", "nova_group"]],
    left_on="off_code", right_on="code", how="left").drop(columns="code")

print(matched.shape, "missing ingredient text:", matched["ingredients_text"].isna().sum())
print(matched["nova"].value_counts().sort_index().tolist())   

matched.to_csv("/kaggle/working/fpx_off_matched.csv", index=False)

off_code unique: True
(1884, 20) missing ingredient text: 0
[477, 61, 265, 1081]
